# 02 — Fine-tune Flan-T5 with LoRA (PEFT)
Transfer learning: freeze base weights, train only low-rank adapter matrices.
Uses `google/flan-t5-base` + PEFT LoRA + HuggingFace Trainer.

In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes mlflow torch --quiet

In [ ]:
import os
import mlflow
import torch
import numpy as np
from pathlib import Path
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

BASE_MODEL = 'google/flan-t5-base'
DATA_DIR   = Path('../backend/data')
MODEL_DIR  = Path('../backend/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load Dataset & Tokenizer

In [ ]:
dataset = load_from_disk(str(DATA_DIR / 'processed_dataset'))
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 512

def tokenize(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding='max_length',
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch['target_text'],
            max_length=MAX_TARGET_LEN,
            truncation=True,
            padding='max_length',
        )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

tokenized = dataset.map(tokenize, batched=True, remove_columns=['input_text', 'target_text'])
print(tokenized)

## 2. Load Base Model & Apply LoRA

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,                              # rank — higher = more capacity
    lora_alpha=32,                     # scaling factor
    lora_dropout=0.1,
    target_modules=['q', 'v'],         # attention projection matrices
    bias='none',
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Trainable params should be ~0.1% of total — that's the point of LoRA

## 3. Configure Training

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR / 'checkpoints'),
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=3e-4,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=10,
    report_to=[],  # switch to 'mlflow' if MLflow server is running
    fp16=(DEVICE == 'cuda'),
    dataloader_num_workers=2,
)

## 4. Train with MLflow Tracking

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

mlflow.set_experiment('immigration-ai-flan-t5-lora')

with mlflow.start_run(run_name='lora-r16') as run:
    mlflow.log_params({
        'base_model': BASE_MODEL,
        'lora_r': lora_config.r,
        'lora_alpha': lora_config.lora_alpha,
        'lora_dropout': lora_config.lora_dropout,
        'target_modules': ','.join(lora_config.target_modules),
        'epochs': training_args.num_train_epochs,
        'batch_size': training_args.per_device_train_batch_size,
        'lr': training_args.learning_rate,
    })

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['validation'],
        tokenizer=tokenizer,
        data_collator=collator,
    )

    trainer.train()

    eval_results = trainer.evaluate(tokenized['test'])
    mlflow.log_metrics({'test_loss': eval_results['eval_loss']})
    print('Test evaluation:', eval_results)

print(f'MLflow Run ID: {run.info.run_id}')

## 5. Save LoRA Adapter

In [ ]:
adapter_path = MODEL_DIR / 'lora_adapter'
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))
print(f'LoRA adapter saved to {adapter_path}')

# List saved files
for f in sorted(adapter_path.iterdir()):
    print(f'  {f.name}')

## 6. Quick Inference Test

In [ ]:
# Reload model from saved adapter to verify
loaded_base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
loaded_model = PeftModel.from_pretrained(loaded_base, str(adapter_path))
loaded_model.eval()

test_question = 'What documents do I need to apply for a green card through marriage?'
prompt = f'Answer the following US immigration question accurately and clearly:\n\nQuestion: {test_question}\n\nAnswer:'

inputs = tokenizer(prompt, return_tensors='pt', max_length=256, truncation=True)
with torch.no_grad():
    outputs = loaded_model.generate(
        **inputs,
        max_new_tokens=300,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f'Q: {test_question}\n')
print(f'A: {answer}')

## Summary
- Fine-tuned `google/flan-t5-base` with LoRA (r=16) on immigration Q&A
- Only ~0.1% of parameters trained — efficient transfer learning
- Adapter saved to `backend/models/lora_adapter/`
- Proceed to `03_evaluate.ipynb` for full evaluation metrics